## ETL dei dati

### Comincio dal dataset guardian_articles

In [1]:
print("Hello, World!")

Hello, World!


In [2]:
# Carico i dati
import pandas as pd

df_guardian = pd.read_csv("guardian_articles.csv", sep=",", encoding="utf-8")

In [2]:
# Informazioni generali sul dataframe
print(df_guardian.dtypes) 
print(df_guardian.shape[0])


article_id            object
sectionName           object
webTitle              object
webUrl                object
bodyContent           object
webPublicationDate    object
id                     int64
dtype: object
149839


In [3]:
# calcolo i valori mancanti per ogni colonna
missing_count_guardian = {}
nan_count_guardian = {}
for col in df_guardian.columns:
    missing_count = df_guardian[col].isnull().sum()
    NaN_count = df_guardian[col].isna().sum()
    missing_count_guardian[col] = missing_count
    nan_count_guardian[col] = NaN_count
    print(f"Colonna: {col}, Valori mancanti: {missing_count}, Valori NaN: {NaN_count}")

# salvo un dataframe con le righe che hanno valori mancanti
df_guardian_missing = df_guardian[df_guardian.isnull().any(axis=1)]

# elimino le righe con valori mancanti
df_guardian = df_guardian.dropna()

Colonna: article_id, Valori mancanti: 0, Valori NaN: 0
Colonna: sectionName, Valori mancanti: 0, Valori NaN: 0
Colonna: webTitle, Valori mancanti: 0, Valori NaN: 0
Colonna: webUrl, Valori mancanti: 0, Valori NaN: 0


Colonna: bodyContent, Valori mancanti: 1108, Valori NaN: 1108
Colonna: webPublicationDate, Valori mancanti: 0, Valori NaN: 0
Colonna: id, Valori mancanti: 0, Valori NaN: 0


In [4]:
# Analizzo le righe duplicate per ogni colonna
duplicate_count_guardian = {}
for col in df_guardian.columns:
    duplicate_count = df_guardian.duplicated(subset=[col]).sum()
    duplicate_count_guardian[col] = duplicate_count
    print(f"Colonna: {col}, Righe duplicate: {duplicate_count}")

# Ispeziono le righe duplicate di webTitle, mostrando solo webTitle e bodyContent
print(df_guardian[df_guardian.duplicated(subset=['webTitle'], keep=False)][['webTitle', 'bodyContent']].sort_values(by='webTitle'))
# Ispeziono le righe duplicate di bodycontent, mostrando solo bodyContent
print(df_guardian[df_guardian.duplicated(subset=['bodyContent'], keep=False)][['bodyContent']].sort_values(by='bodyContent'))

# rimuovo le righe duplicate di bodyContent e articleid, mantenendo solo la prima occorrenza
df_guardian = df_guardian.drop_duplicates(subset=['bodyContent', 'article_id'], keep='first')

Colonna: article_id, Righe duplicate: 11
Colonna: sectionName, Righe duplicate: 148572
Colonna: webTitle, Righe duplicate: 1636
Colonna: webUrl, Righe duplicate: 11


Colonna: bodyContent, Righe duplicate: 291
Colonna: webPublicationDate, Righe duplicate: 28707
Colonna: id, Righe duplicate: 0
                                                 webTitle  \
120348   Children’s books roundup – the best new pictu...   
123517   Children’s books roundup – the best new pictu...   
78690   'What they did to me was so horrific': brutal ...   
78691   'What they did to me was so horrific': brutal ...   
9915     20 of the best Android apps and games this month   
...                                                   ...   
43374            Your best comments on the Guardian today   
44168            Your best comments on the Guardian today   
43158            Your best comments on the Guardian today   
130071    ‘You think wrong!’: top 10 rants in literature    
130070    ‘You think wrong!’: top 10 rants in literature    

                                              bodyContent  
120348  World Book Day tie-in titles are especially st...  
123517  This month’s

                                              bodyContent
72439   91st Academy Awards 1am, Sky Cinema Oscars Eve...
72437   91st Academy Awards 1am, Sky Cinema Oscars Eve...
134218  A picture showing empty shelves was taken at a...
134217  A picture showing empty shelves was taken at a...
107592  Are Covid-19 death rates decreasing? Most stat...
...                                                   ...
49198   ‘I am the world’s best known plumber,” writes ...
49201   ‘I am the world’s best known plumber,” writes ...
134348  • Molly Oldfield hosts Everything Under the Su...
135243  • Molly Oldfield hosts Everything Under the Su...
132925  • Molly Oldfield hosts Everything Under the Su...

[331 rows x 1 columns]


In [5]:
# modifico la colonna WebPublicationDate in formato datetime
df_guardian['webPublicationDate'] = pd.to_datetime(df_guardian['webPublicationDate'], format='%Y-%m-%dT%H:%M:%SZ')
data_min_guardian = df_guardian['webPublicationDate'].min()
data_max_guardian = df_guardian['webPublicationDate'].max()
print("data min: ", data_min_guardian)
print("data max: ", data_max_guardian)

data min:  2016-01-25 20:07:16
data max:  2022-06-30 23:53:09


In [6]:
# Rimuovo le colonne che non mi servono per l'analisi
df_guardian = df_guardian.drop(columns=['article_id', 'webUrl', 'id'])

# Salvo il dataframe pulito in un nuovo file CSV
df_guardian.to_csv("guardian_articles_cleaned.csv", index=False, encoding="utf-8")

In [7]:
# vedo i valori unici della colonna sectionName
tema_guardian = df_guardian['sectionName'].unique()
print(tema_guardian)

['US news' 'World news' 'Football' 'Sport' 'Television & radio'
 'Guardian Small Business Network' 'News' 'Australia news' 'Business'
 'Opinion' 'Politics' 'Education' 'Money' 'UK news' 'Books' 'Media'
 'Life and style' 'Art and design' 'Film' 'Stage' 'Music' 'Environment'
 "Children's books" 'Travel' 'Science' 'Society' 'Technology'
 'Guardian Careers' 'Fashion' 'Guardian Sustainable Business' 'Food'
 'Global' 'Crosswords' 'Community' 'Law' 'Culture' 'Housing Network'
 'Working in development' 'Public Leaders Network' 'Teacher Network'
 'Cities' 'Guardian Masterclasses'
 'Global Development Professionals Network'
 'Healthcare Professionals Network' 'Women in Leadership' 'Media Network'
 'Global development' 'Culture professionals network' 'Games'
 'Voluntary Sector Network' 'Social Care Network' 'Side Hustle'
 'GNM press office' 'Info' 'Membership' 'Guardian Foundation' 'Weather'
 'Help' 'From the Observer' 'Trade Boost' 'From the Guardian' 'Inequality'
 'Sustainability' 'Salesforce: 

In [ ]:
# raccolgo tutte le variabili calcolate durante l'analisi in un unico dizionario,
# cosi' da poterle riutilizzare facilmente in un altro notebook
extra_guardian = {
    "missing_count": missing_count_guardian,
    "nan_count": nan_count_guardian,
    "duplicate_count": duplicate_count_guardian,
    "data_min": data_min_guardian,
    "data_max": data_max_guardian,
    "temi": tema_guardian,
}

#salvo il dizionario in un file json all'interno della cartella del progetto, cosi' da poterlo riutilizzare in un altro notebook



In [9]:
import json
import numpy as np


def _json_default(obj):
    """Converte in tipi standard i valori non nativamente serializzabili in JSON
    (Timestamp di pandas, DataFrame, array/interi/float numpy)."""
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    raise TypeError(f"Tipo non serializzabile in JSON: {type(obj)}")


with open("extra_guardian.json", "w", encoding="utf-8") as f:
    json.dump(extra_guardian, f, ensure_ascii=False, indent=2, default=_json_default)

print("Salvato extra_guardian.json")

Salvato extra_guardian.json
